In [33]:
from __future__ import annotations

from decimal import Decimal, ROUND_HALF_UP
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine


# =========================
# 1) 설정
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA = "d1_machine_log"
TABLES_FCT = ["FCT1_machine_log", "FCT2_machine_log", "FCT3_machine_log", "FCT4_machine_log"]

# ✅ 범위(원하면 바꿔)
# - "2026%"  : 2026년 전체
# - "202601%": 2026년 1월
# - "202602%": 2026년 2월
END_DAY_LIKE = "2026%"

RELEASE = "제품 지그 해제 완료"
ON      = "제품 안착 완료 ON"

MAX_SEC = 500.0  # 500초 초과 제외 (요구사항)


# =========================
# 2) DB 연결
# =========================
def make_engine(cfg: dict) -> Engine:
    url = (
        f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

engine = make_engine(DB_CONFIG)


# =========================
# 3) 유틸
# =========================
def parse_time_to_seconds(t: str) -> Optional[float]:
    """'HH:MI:SS.xx' 또는 'HH:MI:SS' -> 초(float). 파싱 실패 시 None."""
    if t is None:
        return None
    s = str(t).strip()
    if not s:
        return None

    parts = s.split(":")
    if len(parts) != 3:
        return None

    try:
        hh = int(parts[0])
        mm = int(parts[1])
        sec_part = parts[2]
        if "." in sec_part:
            ss_str, frac_str = sec_part.split(".", 1)
            ss = int(ss_str)
            frac = float("0." + "".join(ch for ch in frac_str if ch.isdigit()))
        else:
            ss = int(sec_part)
            frac = 0.0

        return hh * 3600 + mm * 60 + ss + frac
    except Exception:
        return None


def round_half_up(x: float, ndigits: int = 2) -> float:
    q = Decimal("1").scaleb(-ndigits)
    return float(Decimal(str(x)).quantize(q, rounding=ROUND_HALF_UP))


def tukey_five_number(values: List[float]) -> Dict[str, Optional[float]]:
    """
    lower_outlier, q1, median, q3, upper_outlier (whisker)
    + upper_outlier_max + upper_outlier_range("whisker~max_outlier")
    """
    if not values:
        return {
            "lower_outlier": None, "q1": None, "median": None, "q3": None, "upper_outlier": None,
            "upper_outlier_max": None, "upper_outlier_range": None
        }

    arr = np.asarray(values, dtype=float)
    arr.sort()

    q1  = float(np.quantile(arr, 0.25, method="linear"))
    med = float(np.quantile(arr, 0.50, method="linear"))
    q3  = float(np.quantile(arr, 0.75, method="linear"))
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    inlier = arr[(arr >= lower_fence) & (arr <= upper_fence)]
    if inlier.size == 0:
        lower_w = float(arr.min())
        upper_w = float(arr.max())
    else:
        lower_w = float(inlier.min())
        upper_w = float(inlier.max())

    upper_outliers = arr[arr > upper_w]
    upper_outlier_max = float(upper_outliers.max()) if upper_outliers.size > 0 else None

    lower_w_r = round_half_up(lower_w, 2)
    q1_r      = round_half_up(q1, 2)
    med_r     = round_half_up(med, 2)
    q3_r      = round_half_up(q3, 2)
    upper_w_r = round_half_up(upper_w, 2)

    if upper_outlier_max is not None:
        upper_outlier_max_r = round_half_up(upper_outlier_max, 2)
        upper_range = f"{upper_w_r:.2f}~{upper_outlier_max_r:.2f}"
    else:
        upper_outlier_max_r = None
        upper_range = f"{upper_w_r:.2f}~{upper_w_r:.2f}"

    return {
        "lower_outlier": lower_w_r,
        "q1": q1_r,
        "median": med_r,
        "q3": q3_r,
        "upper_outlier": upper_w_r,
        "upper_outlier_max": upper_outlier_max_r,
        "upper_outlier_range": upper_range,
    }


# =========================
# 4) 데이터 로드 (여러 달 포함)
# =========================
def fetch_events(engine: Engine, schema: str, table: str, end_day_like: str) -> pd.DataFrame:
    fqn = f'"{schema}"."{table}"'

    sql = text(f"""
        SELECT
            end_day,
            end_time,
            contents
        FROM {fqn}
        WHERE end_day LIKE :end_day_like
          AND contents IN (:release, :on)
          AND end_time IS NOT NULL
        ORDER BY end_day ASC, end_time ASC
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={
                "end_day_like": end_day_like,
                "release": RELEASE,
                "on": ON
            }
        )
    return df


# =========================
# 5) 페어링 계산 (절대시간 + month 컬럼 생성)
# =========================
def compute_swap_pairs_month_safe(df_events: pd.DataFrame) -> List[Dict[str, object]]:
    """
    end_day(YYYYMMDD) + end_time(HH:MI:SS.xx) -> 절대시간(t_abs)으로 정렬/페어링
    RELEASE -> next ON 페어링
    delta = ON - RELEASE (초)

    반환: [{"month": "YYYYMM", "swap_sec": float}, ...]
    - month는 "ON 이벤트의 end_day[:6]" 기준 (예: 20260201 -> 202602)
    """
    df = df_events.copy()

    df["t_day_sec"] = df["end_time"].map(parse_time_to_seconds)
    df = df.dropna(subset=["t_day_sec"])

    df["d"] = pd.to_datetime(df["end_day"], format="%Y%m%d", errors="coerce")
    df = df.dropna(subset=["d"])

    df["t_abs"] = (df["d"].astype("int64") // 10**9) + df["t_day_sec"]
    df = df.sort_values("t_abs", ascending=True).reset_index(drop=True)

    pending_release: Optional[float] = None
    out: List[Dict[str, object]] = []

    for _, r in df.iterrows():
        c = r["contents"]
        t = float(r["t_abs"])

        if c == RELEASE:
            pending_release = t

        elif c == ON:
            if pending_release is None:
                continue

            delta = t - pending_release
            pending_release = None

            if delta <= 0:
                continue
            if delta > MAX_SEC:
                continue

            end_day_str = str(r["end_day"])
            month = end_day_str[:6]  # ✅ 예: 20260201 -> 202602

            out.append({
                "month": month,
                "swap_sec": round_half_up(delta, 2)
            })

    return out


# =========================
# 6) 실행: df_all(원시) + df_summary(월별 요약)
# =========================
all_rows: List[Dict[str, object]] = []
per_table_month_values: Dict[tuple[str, str], List[float]] = {}  # (table, month) -> values

for t in TABLES_FCT:
    df_e = fetch_events(engine, SCHEMA, t, END_DAY_LIKE)
    pairs = compute_swap_pairs_month_safe(df_e)

    for p in pairs:
        all_rows.append({"table": t, "month": p["month"], "swap_sec": p["swap_sec"]})

df_all = pd.DataFrame(all_rows)

# ✅ (table, month)별 리스트 구성
if not df_all.empty:
    for (table, month), g in df_all.groupby(["table", "month"], sort=True):
        per_table_month_values[(table, month)] = g["swap_sec"].tolist()

# ✅ 월별 요약(df_summary): table별 + ALL_FCT(월별 통합)
summary_rows: List[Dict[str, object]] = []

# table별
for (table, month), vals in sorted(per_table_month_values.items(), key=lambda x: (x[0][1], x[0][0])):
    s = tukey_five_number(vals)
    summary_rows.append({"month": month, "table": table, "n": len(vals), **s})

# 월별 통합(ALL_FCT)
if not df_all.empty:
    for month, g in df_all.groupby("month", sort=True):
        vals = g["swap_sec"].tolist()
        s = tukey_five_number(vals)
        summary_rows.append({"month": month, "table": "ALL_FCT", "n": len(vals), **s})

df_summary = pd.DataFrame(summary_rows)

# 컬럼 순서 정렬
cols = [
    "month", "table", "n",
    "lower_outlier", "q1", "median", "q3", "upper_outlier",
    "upper_outlier_max", "upper_outlier_range"
]
df_summary = df_summary[[c for c in cols if c in df_summary.columns]]

# 출력(Jupyter)
display(df_all)
display(df_summary)

,table,month,swap_sec
0,FCT1_machine_log,202601,4.91
1,FCT1_machine_log,202601,4.93
2,FCT1_machine_log,202601,23.59
3,FCT1_machine_log,202601,4.85
4,FCT1_machine_log,202601,4.85
...,...,...,...
150555,FCT4_machine_log,202602,4.18
150556,FCT4_machine_log,202602,4.23
150557,FCT4_machine_log,202602,28.34
150558,FCT4_machine_log,202602,4.31


,month,table,n,lower_outlier,q1,median,q3,upper_outlier,upper_outlier_max,upper_outlier_range
0,202601,FCT1_machine_log,35749,4.09,4.99,5.11,7.44,11.11,498.83,11.11~498.83
1,202601,FCT2_machine_log,33946,4.27,4.95,5.05,7.18,10.52,497.08,10.52~497.08
2,202601,FCT3_machine_log,37843,4.00,4.36,4.45,5.61,7.48,491.70,7.48~491.70
3,202601,FCT4_machine_log,38417,2.79,4.21,4.29,5.22,6.73,494.60,6.73~494.60
4,202602,FCT1_machine_log,1161,4.86,5.04,5.13,5.84,7.02,420.18,7.02~420.18
5,202602,FCT2_machine_log,963,4.79,4.97,5.04,5.13,5.33,407.50,5.33~407.50
6,202602,FCT3_machine_log,1235,4.14,4.36,4.45,5.40,6.92,269.63,6.92~269.63
7,202602,FCT4_machine_log,1246,4.02,4.20,4.25,4.32,4.50,301.58,4.50~301.58
8,202601,ALL_FCT,145955,2.79,4.37,4.97,6.35,9.31,498.83,9.31~498.83
9,202602,ALL_FCT,4605,4.02,4.33,4.97,5.18,6.45,420.18,6.45~420.18


In [34]:
from sqlalchemy import text

SAVE_SCHEMA = "g_production_film"
SAVE_TABLE  = "fct_op_criteria"

def ensure_schema_table(engine):
    with engine.begin() as conn:
        # schema
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{SAVE_SCHEMA}";'))

        # table
        conn.execute(text(f"""
        CREATE TABLE IF NOT EXISTS "{SAVE_SCHEMA}"."{SAVE_TABLE}" (
            month text NOT NULL,
            "table" text NOT NULL,
            n bigint,
            lower_outlier double precision,
            q1 double precision,
            median double precision,
            q3 double precision,
            upper_outlier double precision,
            upper_outlier_max double precision,
            upper_outlier_range text,
            updated_at timestamptz NOT NULL DEFAULT now(),
            CONSTRAINT "{SAVE_TABLE}__uq" UNIQUE (month, "table")
        );
        """))

def upsert_df_summary(engine, df_summary: pd.DataFrame):
    # 저장 컬럼만 정확히 맞춤
    cols = [
        "month", "table", "n",
        "lower_outlier", "q1", "median", "q3", "upper_outlier",
        "upper_outlier_max", "upper_outlier_range"
    ]
    df = df_summary.copy()[cols]

    # NaN -> None (DB NULL)
    df = df.where(pd.notnull(df), None)

    sql = text(f"""
    INSERT INTO "{SAVE_SCHEMA}"."{SAVE_TABLE}" (
        month, "table", n,
        lower_outlier, q1, median, q3, upper_outlier,
        upper_outlier_max, upper_outlier_range,
        updated_at
    )
    VALUES (
        :month, :table, :n,
        :lower_outlier, :q1, :median, :q3, :upper_outlier,
        :upper_outlier_max, :upper_outlier_range,
        now()
    )
    ON CONFLICT (month, "table") DO UPDATE SET
        n = EXCLUDED.n,
        lower_outlier = EXCLUDED.lower_outlier,
        q1 = EXCLUDED.q1,
        median = EXCLUDED.median,
        q3 = EXCLUDED.q3,
        upper_outlier = EXCLUDED.upper_outlier,
        upper_outlier_max = EXCLUDED.upper_outlier_max,
        upper_outlier_range = EXCLUDED.upper_outlier_range,
        updated_at = now();
    """)

    rows = df.to_dict(orient="records")
    with engine.begin() as conn:
        conn.execute(sql, rows)

# ---- 실행 ----
ensure_schema_table(engine)
upsert_df_summary(engine, df_summary)

print(f"[OK] Saved df_summary -> {SAVE_SCHEMA}.{SAVE_TABLE} (rows={len(df_summary)})")

[OK] Saved df_summary -> g_production_film.fct_op_criteria (rows=10)
